In [ ]:
from osgeo import gdal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Read the mangrove extent data
dataset = gdal.Open('../../data/supplementary_figures/extent/GMW_v3_2020/gmw_v3_2020_0.25d.tif', gdal.GA_ReadOnly)
band = dataset.GetRasterBand(1) # 波段序号从1开始，而不是0
image_data = band.ReadAsArray()
image_data[image_data == 0] = np.nan
geo_transform = dataset.GetGeoTransform()
top_left_x = geo_transform[0]
pixel_width = geo_transform[1]
top_left_y = geo_transform[3]
pixel_height = geo_transform[5]
height, width = image_data.shape
lon_image = np.linspace(top_left_x, top_left_x + pixel_width * width, width)
lat_image = np.linspace(top_left_y, top_left_y + pixel_height * height, height)

# Read the observation data
data = pd.read_csv('../../data/supplementary_figures/obs/mangrove_db_AGB/mangrove_db_AGB.csv')
lat = data.iloc[:, 0]
lon = data.iloc[:, 1]

plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'

fig, ax = plt.subplots(figsize=(12, 3), dpi=800, subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
ax.set_extent([-180, 180, -40, 40])
ax.add_feature(cfeature.LAND, facecolor='lightgray')

# Plot the mangrove extent
green_cmap = ListedColormap(['green']) 
extent = ax.imshow(image_data, cmap= green_cmap, vmin=0, vmax=1, origin='upper',
                   extent=(lon_image.min(), lon_image.max(), lat_image.min(), lat_image.max()))
# Plot the observation sites
scatter = ax.scatter(lon, lat, color='white', s=20, edgecolors='k', linewidths=0.5, label='Sites of observations')

# Add Legend
ax.legend(['Sites of observations'], loc = [0.01, 0.2], frameon=False, markerscale=2)
cbar_ax = fig.add_axes([0.02, 0.15, 0.04, 0.05])
cbar = plt.colorbar(extent, cax=cbar_ax, orientation='horizontal')
cbar.ax.set_xticklabels([])
cbar.ax.tick_params(labelsize=14, length=0)
cbar.outline.set_visible(False)
cbar.ax.text(1.3, 0.5, 'Mangrove extent in 2020', va='center', ha='left', transform=cbar.ax.transAxes)

plt.savefig('../../figures/supplementary/figS15_field_observation_distribution.png', dpi=800)
plt.show()